# **Montamos el drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Archivos de las clases**

Cada celda a continuacion crea un archivo .py con una clase limitada a su tarea

In [ ]:

%%writefile /content/drive/MyDrive/ProyectoFin/preprocesador.py

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

class preprocesador:
  def __init__(self, ruta_csv):
    #Atributos creo
    self.ruta = ruta_csv
    self.df = None
    self.df_preprocesado = None
    self.preprocessor = None

#Setter creo para la BD
  def cargar_BD (self, dataframe):
    self.df = dataframe
    print(f"Datos cargados desde la base de datos: {self.df.shape}")

  #Metodos


  def cargar_CSV(self):
    try:
      self.df = pd.read_csv(self.ruta, sep=";")
      self.df = self.df.drop("id")

      print("Datos cargados")
      print(self.df.shape)
    except FileNotFoundError:
      print("No se encontro el archivo")

  def limpiar_datos(self):
    temp_df = self.df.copy()
    antes = len(temp_df)

    temp_df = temp_df.dropna()
    temp_df = temp_df.drop_duplicates()

    if "age" in temp_df.columns:
      temp_df["age"] = (temp_df["age"] / 365).astype(int)

#Filtrado por presiones
    temp_df = temp_df[(temp_df["ap_hi"] <= 250) & (temp_df["ap_hi"] >= 40)]
    temp_df = temp_df[(temp_df["ap_lo"] <= 150) & (temp_df["ap_lo"] >= 30)]
    temp_df = temp_df[(temp_df["ap_hi"]>temp_df["ap_lo"])]
    print(f"Los registros despues de filtrar por ap: {temp_df.shape}")


    #Calcular el imc
    if "weight" in temp_df.columns and "height" in temp_df.columns:
      temp_df["imc"] = temp_df["weight"] / ((temp_df["height"] / 100) ** 2)



#filtrado por datos(peso,altura y imc)
    temp_df = temp_df[(temp_df["weight"] <= 250) & (temp_df["weight"] >= 35)]
    temp_df = temp_df[(temp_df["height"] <= 230) & (temp_df["height"] >= 130)]
    temp_df = temp_df[(temp_df["imc"] <= 55) & (temp_df["imc"] >= 12)]
    print(f"Los registros despues de filtrar por peso y altura: {temp_df.shape}")

    despues = len(temp_df)
    print(f"Registros eliminados: {antes - despues}")

    self.df_limpio = temp_df

  def transformar_datos(self,df_muestreado, target='cardio'):


    #Features and Target
    colums_ign = [target, "height", "weight"]
    x = df_muestreado.drop(colums_ign, axis=1)
    y = df_muestreado[target]

    #Split data
    x_temp, x_test, y_temp, y_test = train_test_split(x, y, test_size=0.2, random_state=42,stratify= y)
    x_train, x_val, y_train, y_val = train_test_split(x_temp, y_temp, test_size=0.25, random_state=42, stratify= y_temp)

    #Codificacion y escalado de datos
    scaler = StandardScaler()
    codificador = OneHotEncoder(drop="first", sparse_output=False)

    #Columnas a procesar
    cols_num =["age", "ap_hi", "ap_lo", "imc"]
    cols_cat = ["gender", "cholesterol", "gluc"]
    cols_bin = ["smoke", "alco","active"]

    self.preprocessor = ColumnTransformer(
        transformers=[
        ('num', scaler, cols_num),
        ('cat', codificador, cols_cat),
        ('bin', 'passthrough', cols_bin)
        ],

        remainder="drop")

    #x_train_procesado = self.preprocessor.fit_transform(x_train)
    #x_test_procesado = self.preprocessor.transform(x_test)
    #x_test_procesado = self.preprocessor.transform(x_val)

    #Colums = self.preprocessor.get_feature_names_out()

    return x_train, y_train, x_val, y_val, x_test, y_test

  def obtener_datos (self):
    return self.df_limpio

  def obtener_preprocessor(self):
    return self.preprocessor



Overwriting /content/drive/MyDrive/ProyectoFin/preprocesador.py


In [ ]:
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 62.3 MB/s eta 0:00:00


In [ ]:
%%writefile /content/drive/MyDrive/ProyectoFin/gestorDB.py

import mysql.connector
import pandas as pd
from mysql.connector import Error

class gestorDB:
  def __init__(self):
    self.config={
      "host": "2ps03y.h.filess.io",
      "user": "Heartcare_instantmud",
      "password":  "f226c276a2e6c7aac3dc37870d32a781de3dd29d",
      "database": "Heartcare_instantmud",
      "port": "61002"
    }

  def obtener_datos(self, vista = "dataset_completo"):
    query = f"SELECT * FROM {vista}"
    connection = None
    try:
      connection = mysql.connector.connect(**self.config)
      df = pd.read_sql(query,connection)
      print(f"Datos cargados correctamente desde la vista: {vista}")
      return df

    except Error as e:
      print(f"Errror al conectar a la base de datos: {e}")
    finally:
      if connection and connection.is_connected():
        connection.close()

Overwriting /content/drive/MyDrive/ProyectoFin/gestorDB.py


In [ ]:
%%writefile /content/drive/MyDrive/ProyectoFin/ModelosML.py

import sklearn

Writing /content/drive/MyDrive/ProyectoFin/ModelosML.py
